In [ ]:
%pip install datasets
%pip install dotenv

In [59]:
# Initial setup
from datasets import load_dataset
import dotenv
import os
import requests
import pprint

dotenv.load_dotenv()
HUGGINGFACE_API_KEY = os.getenv("HUGGINGFACE_API_KEY")
ALINIA_STAGING_API_KEY = os.getenv("ALINIA_STAGING_API_KEY")

In [67]:
class Results:
    def __init__(self):
        self.tp = 0
        self.tn = 0
        self.fp = 0
        self.fn = 0
        self.accuracy = 0
        self.precision = 0
        self.recall = 0
        self. f1_score = 0
    
    def calculate_stats(self):
        if self.tp > 0 and self.fp > 0:
            self.accuracy = (self.tp + self.tn) / (self.tp + self.tn + self.fp + self.fn)
        else:
            self.accuracy = None

        if self.tp > 0:
            self.precision = self.tp / (self.tp + self.fp)
            self.recall = self.tp / (self.tp + self.fn)
        else:
            self.precision = None
            self.recall = None

        if self.precision is not None and self.recall is not None:
            self.f1_score = 2 * (self.precision * self.recall) / (self.precision + self.recall)
        else:
            self.f1_score = None

    def __str__(self):
        return f"tp: {self.tp}" + "\n" + f"tn: {self.tn}" + "\n" + f"fp: {self.fp}"+ "\n" + f"fn: {self.fn}"+ "\n" + f"accuracy: {self.accuracy}"+ "\n" + f"precision: {self.precision}"+ "\n" + f"recall: {self.recall}"+ "\n" + f"f1_score: {self.f1_score}"

In [69]:
# Helper functions
def get_detection_config_json(
        violence: bool = False, 
        hate: bool = False, 
        wrongdoing: bool = False,
        sexual: bool = False) -> dict:
    
    detection_config_json = {}

    # Set the safety guard flags
    if violence or hate or wrongdoing or sexual:
        detection_config_json["safety"] = {}

        if violence: detection_config_json["safety"]["violence"] = True
        if hate:  detection_config_json["safety"]["hate"] = True
        if wrongdoing:  detection_config_json["safety"]["wrongdoing"] = True
        if sexual:  detection_config_json["safety"]["sexual"] = True
    
    return detection_config_json

def get_alinia_input_json(input_str: str, detection_config: dict) -> dict:
    return {
        "input": input_str,
        "detection_config": detection_config
    }


def evaluate(input_json : dict) -> dict:
    response = requests.post("https://staging.api.alinia.ai/moderations/",
        headers = {"Authorization": f"Bearer {ALINIA_STAGING_API_KEY}"},
        json = input_json,
    )
    return response.json()

def evaluate_hate_1(example, detection_config, results, results2):
    input_str = example["tweet"]
    label = example["class"]

    print(f"Evaluating: {input_str}")
    input_json = get_alinia_input_json(input_str, detection_config)
    response_json = evaluate(input_json)
    
    errors = []
    is_flagged = False

    if 'result' in response_json.keys():
        if 'flagged' in response_json['result'].keys():
            is_flagged = response_json['result']['flagged']
        else:
            errors.append(f'>>>> Missing result, flagged for: {example}')
            print(f'>>>> Missing result.flagged for: {example}')
    else:
        errors.append(f'>>>> Missing result for: {example}')
        print(f'>>>> Missing result for: {example}')

    # pprint.pp(response_json)
    # print(f"is_flagged: {is_flagged}, label: {label}")

    # Hate label only
    if is_flagged and label == 0:
        results.tp += 1
    elif is_flagged and label > 0:
        results.fp += 1
    elif not is_flagged and label > 0:
        results.tn += 1
    elif not is_flagged and label == 0:
        results.fn += 1

    # Hate or Offensive label
    if is_flagged and (label == 0 or label == 1):
        results2.tp += 1
    elif is_flagged and label == 2:
        results2.fp += 1
    elif not is_flagged and label == 2:
        results2.tn += 1
    elif not is_flagged and (label == 0 or label == 1):
        results2.fn += 1

In [62]:
# Load hate datasets

# Columns:
# id: id
# count: number of annotators
# hate_speech_count: number of annotators who judged the tweet to be hate speech
# offensive_language_count: number of annotators who judged the tweet to be offensive
# neither_count: number of users who judged the tweet to be neither offensive nor non-offensive
# class: class label for majority of annotators (0: 'hate-speech', 1: 'offensive-language' or 2: 'neither')
# tweet: The content of the tweet
dataset_hate_1 = load_dataset("tdavidson/hate_speech_offensive", token = HUGGINGFACE_API_KEY)

In [73]:
# Evaluate Safety Guard
HATE_CONFIG = get_detection_config_json(hate = True)

small_ds = dataset_hate_1["train"].select(range(200))
results_hate_1 = Results()
results_hate_2 = Results()

small_ds.map(lambda example: evaluate_hate_1(example, HATE_CONFIG, results_hate_1, results_hate_2))

results_hate_1.calculate_stats()
print("\nResults (Hate Label Only): ")
print(results_hate_1)

results_hate_2.calculate_stats()
print("\nResults (Hate + Offensive Language Labels): ")
print(results_hate_2)

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Evaluating: !!! RT @mayasolovely: As a woman you shouldn't complain about cleaning up your house. &amp; as a man you should always take the trash out...


Map:   0%|          | 1/200 [00:00<03:10,  1.04 examples/s]

Evaluating: !!!!! RT @mleew17: boy dats cold...tyga dwn bad for cuffin dat hoe in the 1st place!!


Map:   1%|          | 2/200 [00:01<02:09,  1.52 examples/s]

...

Evaluating: "@MvckFadden: "Stay beautiful you bitch""


Map:  99%|█████████▉| 198/200 [01:55<00:01,  1.66 examples/s]

Evaluating: "@NICKIMINAJ: #WutKinda

r purple. Ceeeleee"man this gurl was jus playin on the "stupid hoe " track. But in still shitted on sum gurls


Map: 100%|██████████| 200/200 [01:57<00:00,  1.70 examples/s]

Evaluating: "@NastyCopper: Money getting taller and bitches getting blurry"

Results (Hate Label Only): 
tp: 4
tn: 12
fp: 184
fn: 0
accuracy: 0.08
precision: 0.02127659574468085
recall: 1.0
f1_score: 0.04166666666666667

Results (Hate + Offensive Language Labels): 
tp: 175
tn: 11
fp: 13
fn: 1
accuracy: 0.93
precision: 0.9308510638297872
recall: 0.9943181818181818
f1_score: 0.9615384615384616
